<a href="https://colab.research.google.com/github/nevita275/ML_20_Nevita/blob/main/JS03/JS03-Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS
# Langkah 0 - Import Library & Load Data

In [1]:
from google.colab import files
uploaded = files.upload()

Saving wbc.csv to wbc.csv


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('wbc.csv')
print("Shape awal:", df.shape)
from pprint import pprint
pprint(df.columns.tolist())

Shape awal: (569, 33)
['id',
 'diagnosis',
 'radius_mean',
 'texture_mean',
 'perimeter_mean',
 'area_mean',
 'smoothness_mean',
 'compactness_mean',
 'concavity_mean',
 'concave points_mean',
 'symmetry_mean',
 'fractal_dimension_mean',
 'radius_se',
 'texture_se',
 'perimeter_se',
 'area_se',
 'smoothness_se',
 'compactness_se',
 'concavity_se',
 'concave points_se',
 'symmetry_se',
 'fractal_dimension_se',
 'radius_worst',
 'texture_worst',
 'perimeter_worst',
 'area_worst',
 'smoothness_worst',
 'compactness_worst',
 'concavity_worst',
 'concave points_worst',
 'symmetry_worst',
 'fractal_dimension_worst',
 'Unnamed: 32']


**Data awal 569 baris x 33 kolom. Kolom id (tidak informatif) dan Unnamed: 32 (kolom kosong) dibuang sehingga tersisa 31 kolom (1 target + 30 fitur numerik).**

# 1. Pisahkan Variabel yang Dapat & Tidak Dapat Digunakan

In [8]:
kolom_tidak_terpakai = [col for col in df.columns if col == 'id' or 'Unnamed' in col]
df = df.drop(columns=kolom_tidak_terpakai)

print("Shape setelah drop kolom tidak terpakai:", df.shape)
print("Kolom tersisa:")
from pprint import pprint
pprint(df.columns.tolist())

Shape setelah drop kolom tidak terpakai: (569, 31)
Kolom tersisa:
['diagnosis',
 'radius_mean',
 'texture_mean',
 'perimeter_mean',
 'area_mean',
 'smoothness_mean',
 'compactness_mean',
 'concavity_mean',
 'concave points_mean',
 'symmetry_mean',
 'fractal_dimension_mean',
 'radius_se',
 'texture_se',
 'perimeter_se',
 'area_se',
 'smoothness_se',
 'compactness_se',
 'concavity_se',
 'concave points_se',
 'symmetry_se',
 'fractal_dimension_se',
 'radius_worst',
 'texture_worst',
 'perimeter_worst',
 'area_worst',
 'smoothness_worst',
 'compactness_worst',
 'concavity_worst',
 'concave points_worst',
 'symmetry_worst',
 'fractal_dimension_worst']


**Kolom id dibuang karena hanya identifier (tidak informatif untuk prediksi), dan Unnamed: 32 dibuang karena kosong/tidak berisi data**

# 2. Encoding Kolom "diagnosis"

In [9]:
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print("Mapping label:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df['diagnosis'].value_counts())

Mapping label: {'B': np.int64(0), 'M': np.int64(1)}
diagnosis
0    357
1    212
Name: count, dtype: int64


**Benign (B) → 0, Malignant (M) → 1. Data cukup seimbang: 357 Benign, 212 Malignant.**

#3. Pisahkan X, y, Split, dan Standardisasi

In [10]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (455, 30)
X_test shape: (114, 30)


**Standardisasi dilakukan setelah split dengan scaler hanya di-fit pada data train untuk mencegah data leakage.**

#4. Seleksi Fitur dengan SelectKBest

In [11]:
hasil_k = []
for k in range(1, X.shape[1] + 1):
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    acc = accuracy_score(y_test, y_pred)
    hasil_k.append({'k': k, 'accuracy': acc})

hasil_k_df = pd.DataFrame(hasil_k)
print(hasil_k_df.to_string(index=False))

 k  accuracy
 1  0.929825
 2  0.956140
 3  0.956140
 4  0.956140
 5  0.964912
 6  0.964912
 7  0.964912
 8  0.964912
 9  0.973684
10  0.956140
11  0.973684
12  0.973684
13  0.973684
14  0.982456
15  0.973684
16  0.973684
17  0.973684
18  0.982456
19  0.982456
20  0.982456
21  0.973684
22  0.973684
23  0.973684
24  0.973684
25  0.973684
26  0.973684
27  0.973684
28  0.964912
29  0.964912
30  0.964912


**Diuji k=1 sampai k=30, dengan hasil akurasi terbaik dicapai pada k = 14 (akurasi 0.9825), diikuti k=18/19/20 dengan akurasi sama. Karena k=14 lebih sedikit fitur untuk akurasi tertinggi yang sama, itu adalah pilihan paling efisien.**

#5. Cari k Terbaik & Bangun Pipeline

In [12]:
best_k = int(hasil_k_df.loc[hasil_k_df['accuracy'].idxmax(), 'k'])
best_acc = hasil_k_df['accuracy'].max()
print(f"Jumlah fitur terbaik: k = {best_k}, akurasi = {best_acc:.4f}")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=best_k)),
    ('model', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Akurasi pipeline:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Jumlah fitur terbaik: k = 14, akurasi = 0.9825
Akurasi pipeline: 0.9824561403508771
              precision    recall  f1-score   support

           B       0.97      1.00      0.99        72
           M       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



#6. Lihat Fitur-Fitur Terbaik yang Terpilih

In [13]:
selector_fitted = pipeline.named_steps['selector']
mask = selector_fitted.get_support()
fitur_terpilih = X.columns[mask]

print(f"Fitur-fitur terbaik (k={best_k}):")
for f in fitur_terpilih:
    print("-", f)

Fitur-fitur terbaik (k=14):
- radius_mean
- perimeter_mean
- area_mean
- compactness_mean
- concavity_mean
- concave points_mean
- radius_se
- perimeter_se
- radius_worst
- perimeter_worst
- area_worst
- compactness_worst
- concavity_worst
- concave points_worst


**Pipeline final menghasilkan: Akurasi: 98.25%, Precision/recall Benign: 0.97 / 1.00, Precision/recall Malignant: 1.00 / 0.95.**


#7. Berdasarkan hasil analisa Anda berapa jumlah fitur terbaik yang dapat digunakan? Apa saja fitur tersebut?

**Berdasarkan hasil pengujian SelectKBest, jumlah fitur terbaik adalah k = 14, dengan akurasi tertinggi 98.25%. Ke-14 fitur tersebut adalah:
radius_mean, perimeter_mean, area_mean, compactness_mean, concavity_mean, concave points_mean, radius_se, perimeter_se, radius_worst, perimeter_worst, area_worst, compactness_worst, concavity_worst, concave points_worst. Terlihat bahwa fitur-fitur terkait ukuran (radius, perimeter, area) dan bentuk sel (concavity, concave points, compactness).**